In [3]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta
import time
import requests

from secret_key import KMA_API_KEY


BASE_URL = "https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph-aws2_min"
STATION_ID = 108
DATA_DIR = Path("KMA_raw")

EXPECTED_MINUTES_PER_DAY = 1440
MAX_ATTEMPTS = 5
SLEEP_SEC = 1.0
TIMEOUT_SEC = 600


def build_aws_min_params(
    tm1: str,
    tm2: str,
    stn: int = STATION_ID,
    disp: int = 1,
    help_opt: int = 1,
    auth_key: str = KMA_API_KEY,
) -> dict:
    return {
        "tm1": tm1,
        "tm2": tm2,
        "stn": stn,
        "disp": disp,
        "help": help_opt,
        "authKey": auth_key,
    }


def daterange(start_date: datetime, end_date: datetime):
    curr = start_date
    while curr <= end_date:
        yield curr
        curr += timedelta(days=1)


def download_aws_minute_data_day(
    date_obj: datetime,
    output_dir: Path,
    stn: int = STATION_ID,
) -> Path:
    tm1 = date_obj.strftime("%Y%m%d0000")
    tm2 = date_obj.strftime("%Y%m%d2359")

    params = build_aws_min_params(tm1=tm1, tm2=tm2, stn=stn)
    r = requests.get(BASE_URL, params=params, timeout=TIMEOUT_SEC)
    r.raise_for_status()

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"aws_min_{stn}_{date_obj.strftime('%Y%m%d')}.csv"
    out_path.write_bytes(r.content)

    return out_path


def is_data_line(line: str) -> bool:
    s = line.strip()
    if not s or s.startswith("#") or "," not in s:
        return False
    return s.split(",", 1)[0].strip().isdigit()


def count_data_rows(csv_path: Path) -> int:
    if not csv_path.exists():
        return 0

    n = 0
    with open(csv_path, "r", encoding="cp949", errors="ignore") as f:
        for line in f:
            if is_data_line(line):
                n += 1
    return n


def redownload_until_complete(
    date_obj: datetime,
    max_attempts: int = MAX_ATTEMPTS,
) -> bool:
    for attempt in range(1, max_attempts + 1):
        try:
            path = download_aws_minute_data_day(date_obj, DATA_DIR, STATION_ID)
            n_rows = count_data_rows(path)

            if n_rows == EXPECTED_MINUTES_PER_DAY:
                print(
                    f"[FIXED] {date_obj.strftime('%Y-%m-%d')} "
                    f"attempt {attempt}/{max_attempts}: {n_rows} rows"
                )
                return True

            print(
                f"[RETRY] {date_obj.strftime('%Y-%m-%d')} "
                f"attempt {attempt}/{max_attempts}: {n_rows} rows"
            )
            time.sleep(SLEEP_SEC)

        except Exception as e:
            print(
                f"[ERROR] {date_obj.strftime('%Y-%m-%d')} "
                f"attempt {attempt}/{max_attempts}: {e}"
            )
            time.sleep(SLEEP_SEC)

    print(f"[FAIL] {date_obj.strftime('%Y-%m-%d')} unresolved")
    return False


def check_and_fix_period(start_date_str: str, end_date_str: str):
    start = datetime.strptime(start_date_str, "%Y%m%d")
    end = datetime.strptime(end_date_str, "%Y%m%d")

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    n_ok, n_fixed, n_fail = 0, 0, 0

    for d in daterange(start, end):
        path = DATA_DIR / f"aws_min_{STATION_ID}_{d.strftime('%Y%m%d')}.csv"
        n_rows = count_data_rows(path)

        if n_rows == EXPECTED_MINUTES_PER_DAY:
            print(f"[OK] {d.strftime('%Y-%m-%d')} ({n_rows})")
            n_ok += 1
            continue

        status = "MISSING" if not path.exists() else "SHORT"
        print(f"[{status}] {d.strftime('%Y-%m-%d')} ({n_rows}) → retry")

        if redownload_until_complete(d):
            n_fixed += 1
        else:
            n_fail += 1

    print("\n=== SUMMARY ===")
    print(f"OK   : {n_ok}")
    print(f"FIXED: {n_fixed}")
    print(f"FAIL : {n_fail}")


check_and_fix_period("20240101", "20241231")

[MISSING] 2024-01-01 (0) → retry
[FIXED] 2024-01-01 attempt 1/5: 1440 rows
[MISSING] 2024-01-02 (0) → retry
[FIXED] 2024-01-02 attempt 1/5: 1440 rows
[MISSING] 2024-01-03 (0) → retry
[FIXED] 2024-01-03 attempt 1/5: 1440 rows
[MISSING] 2024-01-04 (0) → retry
[RETRY] 2024-01-04 attempt 1/5: 1061 rows
[FIXED] 2024-01-04 attempt 2/5: 1440 rows
[MISSING] 2024-01-05 (0) → retry
[RETRY] 2024-01-05 attempt 1/5: 1003 rows
[FIXED] 2024-01-05 attempt 2/5: 1440 rows
[MISSING] 2024-01-06 (0) → retry
[RETRY] 2024-01-06 attempt 1/5: 990 rows
[FIXED] 2024-01-06 attempt 2/5: 1440 rows
[MISSING] 2024-01-07 (0) → retry
[RETRY] 2024-01-07 attempt 1/5: 732 rows
[FIXED] 2024-01-07 attempt 2/5: 1440 rows
[MISSING] 2024-01-08 (0) → retry
[RETRY] 2024-01-08 attempt 1/5: 1103 rows
[FIXED] 2024-01-08 attempt 2/5: 1440 rows
[MISSING] 2024-01-09 (0) → retry
[RETRY] 2024-01-09 attempt 1/5: 1059 rows
[FIXED] 2024-01-09 attempt 2/5: 1440 rows
[MISSING] 2024-01-10 (0) → retry
[RETRY] 2024-01-10 attempt 1/5: 1042 rows


In [4]:
from pathlib import Path
import pandas as pd


INPUT_DIR = Path("KMA_raw")
OUTPUT_PATH = Path("aws_min_108_20240101_20241231_merged.xlsx")


def read_aws_min_file(path: Path) -> pd.DataFrame:
    """Parse one KMA AWS minute CSV into a DataFrame."""
    try:
        text = path.read_text(encoding="cp949")
    except UnicodeDecodeError:
        text = path.read_text(encoding="utf-8")

    lines = text.splitlines()
    if len(lines) < 23:
        raise ValueError(f"File too short to parse: {path}")

    header_line = lines[20].lstrip("#").strip()
    column_names = header_line.split()
    column_names.append("FLAG")  # trailing flag column

    df = pd.read_csv(
        path,
        sep=",",
        skiprows=22,
        header=None,
        names=column_names,
        encoding="cp949",
    )
    return df


def parse_kma_time(series: pd.Series) -> pd.Series:
    """Parse KMA YYYYMMDDHHMM into datetime."""
    s = series.astype(str).str.strip().str.replace("=", "", regex=False)

    return pd.to_datetime(s, format="%Y%m%d%H%M", errors="coerce")

def merge_aws_minute_files(
    input_dir: Path = INPUT_DIR,
    output_path: Path = OUTPUT_PATH,
) -> pd.DataFrame:
    """Merge AWS minute CSVs into a single time-ordered Excel file."""
    files = sorted(input_dir.glob("aws_min_*.csv"))
    if not files:
        raise FileNotFoundError(f"No AWS files found in: {input_dir}")

    frames: list[pd.DataFrame] = []
    for path in files:
        try:
            df = read_aws_min_file(path)
            frames.append(df)
            print(f"[OK] parsed {path.name} ({len(df)} rows)")
        except Exception as exc:
            print(f"[ERROR] {path.name}: {exc}")

    if not frames:
        raise RuntimeError("No valid data frames were produced.")

    merged = pd.concat(frames, ignore_index=True)

    if "YYMMDDHHMI" in merged.columns:
        merged["DATETIME"] = parse_kma_time(merged["YYMMDDHHMI"])
        merged = merged.dropna(subset=["DATETIME"])
        merged = merged.sort_values("DATETIME").reset_index(drop=True)

    if "FLAG" in merged.columns:
        merged = merged.drop(columns=["FLAG"])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_excel(output_path, index=False)

    print(f"\n[INFO] merged shape: {merged.shape}")
    print(f"[INFO] saved to: {output_path}")
    return merged


if __name__ == "__main__":
    merge_aws_minute_files()

[OK] parsed aws_min_108_20240101.csv (1441 rows)
[OK] parsed aws_min_108_20240102.csv (1441 rows)
[OK] parsed aws_min_108_20240103.csv (1441 rows)
[OK] parsed aws_min_108_20240104.csv (1441 rows)
[OK] parsed aws_min_108_20240105.csv (1441 rows)
[OK] parsed aws_min_108_20240106.csv (1441 rows)
[OK] parsed aws_min_108_20240107.csv (1441 rows)
[OK] parsed aws_min_108_20240108.csv (1441 rows)
[OK] parsed aws_min_108_20240109.csv (1441 rows)
[OK] parsed aws_min_108_20240110.csv (1441 rows)
[OK] parsed aws_min_108_20240111.csv (1441 rows)
[OK] parsed aws_min_108_20240112.csv (1441 rows)
[OK] parsed aws_min_108_20240113.csv (1441 rows)
[OK] parsed aws_min_108_20240114.csv (1441 rows)
[OK] parsed aws_min_108_20240115.csv (1441 rows)
[OK] parsed aws_min_108_20240116.csv (1441 rows)
[OK] parsed aws_min_108_20240117.csv (1441 rows)
[OK] parsed aws_min_108_20240118.csv (1441 rows)
[OK] parsed aws_min_108_20240119.csv (1441 rows)
[OK] parsed aws_min_108_20240120.csv (1441 rows)
[OK] parsed aws_min_